# Interactive Machine Learning Algorithm Explorer

**Start from the best use case**, then explore algorithm choices, assumptions, code, output data, and a live visualization. This notebook is **standalone**: it does not import any companion project files.

Run all cells once, then use the dropdown and buttons in the last cell. Synthetic data is for education, not model benchmarking. Requires `numpy pandas matplotlib scikit-learn torch ipywidgets`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch


## Algorithm catalog and assumptions

In [ ]:
"""Educational use-case catalog. Expands named LinkedIn algorithms with common counterparts."""
ALGORITHMS = [
('Predict a numeric value','Linear Regression','supervised','Numeric target; approximately additive/linear conditional mean; observations independent when using standard inference; severe collinearity destabilizes coefficients; residual patterns, leverage and heteroscedasticity merit checks.'),
('Predict a yes/no outcome','Logistic Regression','supervised','Binary labels; log-odds approximately linear in inputs; independent observations for standard inference; scaled features and regularization help; probabilities may require calibration.'),
('Predict a numeric value robustly','Random Forest Regressor','supervised','Independent representative rows; enough samples per leaf; cannot reliably extrapolate beyond training targets; temporal leakage and high serving costs need attention.'),
('Classify tabular records','Random Forest Classifier','supervised','Labeled representative examples; correlated predictors accepted; individual predictions less interpretable; imbalance must be handled; large ensembles can be slow.'),
('Predict complex tabular patterns','Gradient Boosting','supervised','Reliable labels and validation split; sensitive to tuning, noise and leakage; early stopping/regularization reduces overfit; tabular structured data is typical.'),
('Classify with a maximum margin','Support Vector Machine','supervised','Margin-based classification; features generally scaled; kernel and C tune complexity; large datasets can strain training; scores not calibrated probabilities.'),
('Classify from nearest neighbors','K-Nearest Neighbors','supervised','Distances encode meaningful similarity; features scaled; susceptible to high-dimensional concentration, irrelevant features, and expensive inference.'),
('Model text with word frequencies','Naive Bayes','supervised','Conditional feature independence given class is a simplifying approximation; nonnegative counts for multinomial version; poor calibrated probabilities possible.'),
('Train a small dense neural model','Neural Network (MLP)','supervised','Numeric normalized features; representative labels; sufficiently large training set; nonlinear optimization and tuning; track overfitting and seed variability.'),
('Recognize image-like patterns','Convolutional Neural Network','supervised','Spatially local and repeated patterns; tensors with channel/spatial dimensions; sufficient images or augmentation; distribution shifts can hurt.'),
('Predict sequential signals','Recurrent Neural Network','supervised','Ordered sequence input; temporal signal exists; sequence windows cannot leak future values; plain RNN can struggle with long dependencies.'),
('Learn from token sequences','Transformer Encoder','supervised','Token or time sequence encoded with position information; attention compute rises with sequence length; smaller datasets may not justify complexity.'),
('Group spherical clusters','K-Means','unsupervised','Euclidean distance meaningful after scaling; roughly compact convex clusters; specify k; sensitive to outliers, initializations and uneven density.'),
('Explore nested clusters','Hierarchical Clustering','unsupervised','Chosen distance and linkage represent desired grouping; scaling matters; large N can cost substantial memory and compute.'),
('Find irregular dense clusters','DBSCAN','unsupervised','Density-connected groups separated by sparse areas; eps/min_samples tuned in scaled feature space; varying densities and high dimensions cause trouble.'),
('Detect anomalies through reconstruction','Autoencoder','unsupervised','Train mostly on normal representative data; anomalies reconstruct less well only when learnable; reconstruction error threshold calibrated on validation data.'),
('Reduce correlated features','Principal Component Analysis','reduction','Numeric continuous features; standardize when units differ; linear projections maximize variance, not necessarily task signal; low-variance predictive directions can disappear.'),
]


## Executable Python demonstrations (edit these functions to experiment)

In [ ]:
"""One runnable minimal learning experiment per algorithm, with a visualizable result."""
import numpy as np
from sklearn.datasets import make_regression, make_classification, make_blobs, make_moons
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, mean_squared_error, silhouette_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor,RandomForestClassifier,HistGradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.neural_network import MLPClassifier
from sklearn.cluster import KMeans,AgglomerativeClustering,DBSCAN
from sklearn.decomposition import PCA
import torch
from torch import nn

def run_algorithm(name, seed=42):
    """Returns (dataframe, metric string, chart title) for the selected algorithm."""
    import pandas as pd
    np.random.seed(seed); torch.manual_seed(seed)
    reg=name in ('Linear Regression','Random Forest Regressor')
    if reg:
        X,y=make_regression(n_samples=240,n_features=3,noise=19,random_state=seed)
        model=LinearRegression() if name=='Linear Regression' else RandomForestRegressor(n_estimators=60,random_state=seed)
        a,b=train_test_split(np.arange(len(y)),test_size=.25,random_state=seed)
        model.fit(X[a],y[a]); pred=model.predict(X[b]); score=f'Test RMSE: {np.sqrt(mean_squared_error(y[b],pred)):.3f}'
        return pd.DataFrame({'x':y[b], 'y':pred,'split':'test'}),score,'Actual vs predicted (ideal diagonal)'
    if name in ('K-Means','Hierarchical Clustering','DBSCAN','Principal Component Analysis','Autoencoder'):
        if name=='DBSCAN': X,_=make_moons(n_samples=260,noise=.07,random_state=seed)
        else: X,_=make_blobs(n_samples=260,centers=3,cluster_std=1.2,random_state=seed)
        X=StandardScaler().fit_transform(X)
        if name=='K-Means': labels=KMeans(n_clusters=3,n_init=10,random_state=seed).fit_predict(X)
        elif name=='Hierarchical Clustering': labels=AgglomerativeClustering(n_clusters=3).fit_predict(X)
        elif name=='DBSCAN': labels=DBSCAN(eps=.24,min_samples=5).fit_predict(X)
        elif name=='Principal Component Analysis':
            rng=np.random.default_rng(seed); X=np.column_stack([X,X[:,0]*.9+.1*rng.normal(size=len(X))]); z=PCA(n_components=2).fit_transform(X)
            return pd.DataFrame({'x':z[:,0],'y':z[:,1],'group':0}), 'Reduced 3 numeric features to 2 PCs','PCA projection'
        else:
            inp=torch.tensor(X,dtype=torch.float32); net=nn.Sequential(nn.Linear(2,4),nn.Tanh(),nn.Linear(4,1),nn.Tanh(),nn.Linear(1,4),nn.Tanh(),nn.Linear(4,2))
            opt=torch.optim.Adam(net.parameters(),lr=.02)
            for _ in range(100):
                opt.zero_grad(); pred=net(inp); loss=((pred-inp)**2).mean(); loss.backward();opt.step()
            with torch.no_grad(): recon=net(inp).numpy()
            err=np.sum((recon-X)**2,axis=1)
            return pd.DataFrame({'x':X[:,0],'y':X[:,1],'group':err}),f'Average squared reconstruction error: {np.mean(err):.4f}','Reconstruction error (continuous color)'
        groups=len(set(labels)-{-1}); metric=f'{groups} clusters found; noise label -1 when applicable'
        return pd.DataFrame({'x':X[:,0],'y':X[:,1],'group':labels}),metric,'Clusters in feature space'
    if name in ('Convolutional Neural Network','Recurrent Neural Network','Transformer Encoder'):
        n=160
        if name=='Convolutional Neural Network':
            X=torch.randn(n,1,8,8); y=(X[:,:,2:6,2:6].mean(dim=(1,2,3))>0).long()
            net=nn.Sequential(nn.Conv2d(1,4,3,padding=1),nn.ReLU(),nn.Flatten(),nn.Linear(4*8*8,2))
        else:
            X=torch.randn(n,8,3); y=(X[:,4:,0].mean(dim=1)>0).long()
            if name=='Recurrent Neural Network':
                class RNNModel(nn.Module):
                    def __init__(self): super().__init__();self.r=nn.RNN(3,12,batch_first=True);self.head=nn.Linear(12,2)
                    def forward(self,x): out,_=self.r(x);return self.head(out[:,-1,:])
                net=RNNModel()
            else:
                class TransformerModel(nn.Module):
                    def __init__(self):
                        super().__init__();self.proj=nn.Linear(3,12);self.pos=nn.Parameter(torch.zeros(1,8,12));self.attn=nn.TransformerEncoderLayer(d_model=12,nhead=3,batch_first=True,dropout=0.);self.head=nn.Linear(12,2)
                    def forward(self,x):return self.head(self.attn(self.proj(x)+self.pos)[:,4:,:].mean(dim=1))
                net=TransformerModel()
        train=torch.arange(120); test=torch.arange(120,160); opt=torch.optim.Adam(net.parameters(),lr=.02)
        for _ in range(45):
            net.train();opt.zero_grad();out=net(X[train]);loss=nn.functional.cross_entropy(out,y[train]);loss.backward();opt.step()
        net.eval()
        with torch.no_grad(): logits=net(X[test]); probs=logits.softmax(dim=1)[:,1].numpy()
        acc=accuracy_score(y[test].numpy(),probs>=.5)
        return pd.DataFrame({'x':np.arange(len(probs)),'y':probs,'group':y[test].numpy()}),f'Holdout accuracy: {acc:.2%} (toy dataset)','Predicted class-1 probabilities by sample'
    X,y=make_classification(n_samples=260,n_features=8,n_informative=5,n_redundant=1,random_state=seed)
    X=StandardScaler().fit_transform(X)
    if name=='Logistic Regression': model=LogisticRegression(max_iter=500)
    elif name=='Random Forest Classifier': model=RandomForestClassifier(n_estimators=60,random_state=seed)
    elif name=='Gradient Boosting': model=HistGradientBoostingClassifier(max_iter=75,random_state=seed)
    elif name=='Support Vector Machine': model=SVC(probability=True,random_state=seed)
    elif name=='K-Nearest Neighbors': model=KNeighborsClassifier(n_neighbors=7)
    elif name=='Naive Bayes': model=MultinomialNB();X=np.abs(X)
    elif name=='Neural Network (MLP)': model=MLPClassifier(hidden_layer_sizes=(16,),max_iter=250,random_state=seed)
    else: raise ValueError(f'Unknown algorithm: {name}')
    a,b=train_test_split(np.arange(len(y)),test_size=.25,random_state=seed,stratify=y)
    model.fit(X[a],y[a]); pred=model.predict(X[b]); probs=model.predict_proba(X[b])[:,1]
    return pd.DataFrame({'x':np.arange(len(b)),'y':probs,'group':y[b]}),f'Holdout accuracy: {accuracy_score(y[b],pred):.2%}','Predicted class-1 probabilities by sample'


## Visualization helper

In [ ]:
def plot_demo(df, title):
    fig, ax=plt.subplots(figsize=(8,5))
    if 'group' in df:
        sc=ax.scatter(df.x,df.y,c=df.group,cmap='viridis',alpha=.8);fig.colorbar(sc,ax=ax,label='Group / target / error')
    else:
        ax.scatter(df.x,df.y,alpha=.7); lo=min(df.x.min(),df.y.min());hi=max(df.x.max(),df.y.max());ax.plot([lo,hi],[lo,hi],'--',label='Perfect fit');ax.legend()
    ax.set(xlabel='X / actual / sample index',ylabel='Y / prediction',title=title);fig.tight_layout();plt.show()



## Interactive best-use-case menu

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output, Markdown
import matplotlib.pyplot as plt
import pandas as pd

selector = widgets.Dropdown(
    options=[(f"{case}  →  {name}", name) for case,name,_,_ in ALGORITHMS],
    description="Best use case:",style={"description_width":"initial"},layout=widgets.Layout(width="95%"))
seed = widgets.IntSlider(value=42,min=1,max=1000,description="Random seed:",style={"description_width":"initial"})
assumption_btn=widgets.Button(description="Explain assumptions",button_style="info")
run_btn=widgets.Button(description="Run Python demo",button_style="success")
plot_btn=widgets.Button(description="Run + visualize")
back_btn=widgets.Button(description="Show all use cases")
output=widgets.Output()

def explain(*_):
    with output:
        clear_output(wait=True)
        case,name,family,assumptions=next(row for row in ALGORITHMS if row[1]==selector.value)
        display(Markdown(f"### {name}\n**Best use case:** {case}  \n**Category:** {family}  \n**Assumptions and checks:** {assumptions}"))
        display(Markdown("**What to verify on your data:** Separate train/test sets, prevent leakage, inspect input distributions and class balance (when relevant), benchmark against a simpler baseline, and validate under real deployment conditions."))

def demo(visual=False):
    with output:
        clear_output(wait=True)
        explain_row=next(row for row in ALGORITHMS if row[1]==selector.value)
        display(Markdown(f"### {selector.value}\n**Best use case:** {explain_row[0]}  \n**Assumptions:** {explain_row[3]}"))
        try:
            frame,metric,title=run_algorithm(selector.value,seed.value)
            display(Markdown(f"**Result:** {metric}"))
            display(frame.head(12))
            if visual: plot_demo(frame,title)
        except Exception as exc:print(type(exc).__name__,exc)

def show_all(*_):
    with output:
        clear_output(wait=True)
        display(pd.DataFrame(ALGORITHMS,columns=['Best use case','Algorithm','Family','Assumptions']))
assumption_btn.on_click(explain)
run_btn.on_click(lambda _:demo(False))
plot_btn.on_click(lambda _:demo(True))
back_btn.on_click(show_all)
display(widgets.VBox([selector,seed,widgets.HBox([assumption_btn,run_btn,plot_btn,back_btn]),output]))
explain()


# 12 additional AI algorithms from the visual reference

These lessons extend the existing machine-learning explorer with the 12 algorithms shown in the supplied infographic. Each section includes a short explanation and a compact Python example.

**Important scope notes**
- PPO and SAC cells demonstrate the core policy objectives, not complete reinforcement-learning training systems.
- The DNC cell demonstrates differentiable external-memory reading, not the full allocation/write-controller architecture.
- MCTS uses a small fixed-action reward example to teach selection and backpropagation.
- YOLO requires `pip install ultralytics` and a local image named `sample.jpg`; the first run downloads model weights.
- VGG/ResNet requires `torchvision` (`pip install torchvision`). Its example uses untrained weights so it does not download a checkpoint.
- The remaining examples use Python's standard library or PyTorch already required by this notebook.

Run the cells individually, inspect the tensor shapes and outputs, and then modify the parameters.


## 1. Proximal Policy Optimization (PPO)

PPO is an on-policy policy-gradient method. The clipped objective limits how far a new policy can move from the behavior policy in one update. This compact example computes the clipped surrogate loss; it is not a complete environment/training loop.

In [ ]:
import torch
from torch.distributions import Categorical

# Toy batch: policy logits, sampled actions, old log-probabilities, advantages.
logits = torch.tensor([[1.2, 0.1], [0.2, 0.8], [0.5, 0.4]], requires_grad=True)
actions = torch.tensor([0, 1, 0])
old_log_probs = torch.tensor([-0.35, -0.40, -0.55])
advantages = torch.tensor([1.0, 0.7, -0.3])
clip_eps = 0.2

dist = Categorical(logits=logits)
new_log_probs = dist.log_prob(actions)
ratio = torch.exp(new_log_probs - old_log_probs)
unclipped = ratio * advantages
clipped = torch.clamp(ratio, 1 - clip_eps, 1 + clip_eps) * advantages
policy_loss = -torch.minimum(unclipped, clipped).mean()
policy_loss.backward()
print(f"PPO clipped policy loss: {policy_loss.item():.4f}")
print("Gradient norm:", logits.grad.norm().item())

## 2. A* Search

A* prioritizes nodes by f(n) = g(n) + h(n): cost so far plus an estimated remaining cost. With an admissible heuristic, it can find an optimal path for nonnegative edge costs.

In [ ]:
import heapq

graph = {
    "A": {"B": 1, "C": 4}, "B": {"C": 2, "D": 5},
    "C": {"D": 1}, "D": {}
}
heuristic = {"A": 3, "B": 2, "C": 1, "D": 0}

def astar(graph, start, goal, h):
    frontier = [(h[start], 0, start, [start])]
    best_cost = {start: 0}
    while frontier:
        _, cost, node, path = heapq.heappop(frontier)
        if node == goal:
            return path, cost
        if cost != best_cost[node]:
            continue
        for neighbor, edge_cost in graph[node].items():
            new_cost = cost + edge_cost
            if new_cost < best_cost.get(neighbor, float("inf")):
                best_cost[neighbor] = new_cost
                heapq.heappush(frontier, (new_cost + h[neighbor],
                                          new_cost, neighbor, path + [neighbor]))
    return None, float("inf")

print(astar(graph, "A", "D", heuristic))

## 3. Monte Carlo Tree Search (MCTS)

MCTS repeatedly selects a promising child, expands it, simulates a rollout, and backpropagates the reward. This small single-player tree illustrates UCB selection and reward propagation; production game agents need state transitions and terminal evaluation.

In [ ]:
import math
import random

class Node:
    def __init__(self, reward=None):
        self.reward, self.visits, self.children = reward, 0, []

root = Node()
root.children = [Node(0.2), Node(0.8), Node(0.5)]
for _ in range(500):
    parent = root
    # UCB1 selection among the fixed candidate actions
    log_n = math.log(parent.visits + 1)
    child = max(parent.children, key=lambda c:
                (c.reward if c.reward is not None else 0)
                + math.sqrt(2 * log_n / (c.visits + 1)))
    # Bernoulli rollout using the candidate's reward as its win probability
    reward = 1.0 if random.random() < child.reward else 0.0
    parent.visits += 1
    child.visits += 1
print([(round(c.reward, 2), c.visits) for c in root.children])

## 4. Recurrent Neural Network (RNN) / LSTM

An LSTM uses input, forget, and output gates plus a cell state to preserve information across sequence steps. It is useful for sequential signals, although Transformers often parallelize training more effectively.

In [ ]:
import torch
from torch import nn

torch.manual_seed(7)
lstm = nn.LSTM(input_size=5, hidden_size=16, num_layers=1, batch_first=True)
sequence = torch.randn(4, 12, 5)  # batch, time, features
outputs, (hidden, cell) = lstm(sequence)
print("Sequence output:", outputs.shape)  # [4, 12, 16]
print("Final hidden state:", hidden.shape)

## 5. Differentiable Neural Computer (DNC)

A full DNC includes differentiable external memory, content-based addressing, allocation, and read/write heads. This minimal differentiable memory sketch demonstrates soft addressing and a differentiable read; it is not a complete DNC implementation.

In [ ]:
import torch
from torch import nn
import torch.nn.functional as F

class DifferentiableMemory(nn.Module):
    def __init__(self, slots=8, width=16):
        super().__init__()
        self.memory = nn.Parameter(torch.randn(slots, width) * 0.1)
        self.key = nn.Parameter(torch.randn(width) * 0.1)
    def forward(self):
        weights = F.softmax(self.memory @ self.key, dim=0)  # soft read weights
        return weights @ self.memory, weights

memory = DifferentiableMemory()
read_vector, read_weights = memory()
loss = read_vector.square().mean()
loss.backward()
print("Read vector:", read_vector.shape, "weights sum:", read_weights.sum().item())
print("Memory gradient:", memory.memory.grad.norm().item())

## 6. Alpha-Beta Pruning

Alpha-beta pruning accelerates minimax by skipping branches that cannot affect the final decision. It returns the same choice as minimax when the game tree and move ordering are unchanged.

In [ ]:
def alphabeta(tree, node, depth, alpha, beta, maximizing):
    if depth == 0 or isinstance(tree[node], (int, float)):
        return tree[node]
    if maximizing:
        value = float("-inf")
        for child in tree[node]:
            value = max(value, alphabeta(tree, child, depth-1, alpha, beta, False))
            alpha = max(alpha, value)
            if alpha >= beta: break
        return value
    value = float("inf")
    for child in tree[node]:
        value = min(value, alphabeta(tree, child, depth-1, alpha, beta, True))
        beta = min(beta, value)
        if beta <= alpha: break
    return value

tree = {"root": ["left", "right"], "left": ["a", "b"],
        "right": ["c", "d"], "a": 3, "b": 5, "c": 2, "d": 9}
print("Best minimax value:", alphabeta(tree, "root", 2, float("-inf"),
                                        float("inf"), True))

## 7. YOLO (You Only Look Once)

YOLO is a one-stage object detector that predicts boxes and classes in a single network pass. The example uses Ultralytics YOLO with a small pretrained checkpoint; it downloads weights on first use and needs an image file. Install with `pip install ultralytics`.

In [ ]:
# Optional external-model example: requires ultralytics and a local image.
from ultralytics import YOLO

detector = YOLO("yolo11n.pt")  # pretrained nano model; downloads weights if absent
results = detector.predict(source="sample.jpg", conf=0.25, verbose=False)
for box in results[0].boxes[:5]:
    print("class_id=", int(box.cls[0]), "confidence=", round(float(box.conf[0]), 3),
          "xyxy=", box.xyxy[0].tolist())

## 8. Generative Adversarial Network (GAN)

A GAN trains a generator to create samples and a discriminator to distinguish generated samples from real data. This toy example trains both networks on a one-dimensional Gaussian; it is a teaching demo, not a stable image GAN recipe.

In [ ]:
import torch
from torch import nn

torch.manual_seed(3)
G = nn.Sequential(nn.Linear(4, 16), nn.ReLU(), nn.Linear(16, 1))
D = nn.Sequential(nn.Linear(1, 16), nn.LeakyReLU(0.2), nn.Linear(16, 1), nn.Sigmoid())
opt_g = torch.optim.Adam(G.parameters(), lr=1e-3)
opt_d = torch.optim.Adam(D.parameters(), lr=1e-3)
bce = nn.BCELoss()

for step in range(200):
    real = torch.randn(32, 1) * 0.7 + 2.0
    noise = torch.randn(32, 4)
    fake = G(noise).detach()
    d_loss = bce(D(real), torch.ones(32, 1)) + bce(D(fake), torch.zeros(32, 1))
    opt_d.zero_grad(); d_loss.backward(); opt_d.step()

    generated = G(torch.randn(32, 4))
    g_loss = bce(D(generated), torch.ones(32, 1))
    opt_g.zero_grad(); g_loss.backward(); opt_g.step()

print("Generated sample mean:", G(torch.randn(1000, 4)).mean().item())

## 9. VGGNet / ResNet

VGG stacks small convolutions; ResNet adds identity skip connections so deeper networks can learn residual updates. Torchvision provides reference architectures and pretrained weights; the example uses ResNet-18 with random input and no weight download.

In [ ]:
import torch
from torchvision.models import resnet18, vgg11

# weights=None keeps this example offline and deterministic in terms of setup.
resnet = resnet18(weights=None).eval()
vgg = vgg11(weights=None).eval()
image_batch = torch.randn(2, 3, 224, 224)
with torch.no_grad():
    print("ResNet logits:", resnet(image_batch).shape)
    print("VGG logits:", vgg(image_batch).shape)

## 10. Behavior Trees

Behavior trees organize decisions into composable nodes. Sequence runs children until one fails; selector runs children until one succeeds. They are common in robotics, game AI, and bounded agent workflows.

In [ ]:
SUCCESS, FAILURE, RUNNING = "SUCCESS", "FAILURE", "RUNNING"

class Sequence:
    def __init__(self, *children): self.children = children
    def tick(self, context):
        for child in self.children:
            result = child(context)
            if result != SUCCESS: return result
        return SUCCESS

def has_target(ctx): return SUCCESS if ctx.get("target_visible") else FAILURE
def within_range(ctx): return SUCCESS if ctx.get("distance", 999) < 5 else FAILURE
def engage(ctx):
    ctx["action"] = "engage_target"
    return SUCCESS

tree = Sequence(has_target, within_range, engage)
world = {"target_visible": True, "distance": 3.2}
print("Tree result:", tree.tick(world), "world:", world)

## 11. Soft Actor-Critic (SAC)

SAC is an off-policy actor-critic method for continuous control. It maximizes expected reward plus entropy, encouraging exploration. This example computes a simple entropy-regularized actor objective from a toy Gaussian policy; a full SAC agent also needs twin Q-networks, target networks, and replay buffer.

In [ ]:
import torch
from torch.distributions import Normal

mean = torch.tensor([0.2, -0.1], requires_grad=True)
log_std = torch.tensor([-0.5, -0.3], requires_grad=True)
distribution = Normal(mean, log_std.exp())
action = distribution.rsample()
log_prob = distribution.log_prob(action).sum()
estimated_q = -(action - 0.7).square().sum()  # toy differentiable critic
alpha = 0.2
actor_loss = (alpha * log_prob - estimated_q)
actor_loss.backward()
print("SAC-style actor loss:", actor_loss.item())
print("Policy gradient norm:", torch.stack([mean.grad.norm(), log_std.grad.norm()]).norm().item())

## 12. Attention Mechanisms (Self-Attention & Cross-Attention)

Self-attention lets tokens attend to other tokens in the same sequence. Cross-attention uses queries from one sequence and keys/values from another, as in encoder-decoder models and multimodal fusion.

In [ ]:
import torch
from torch import nn

torch.manual_seed(11)
self_attention = nn.MultiheadAttention(embed_dim=32, num_heads=4, batch_first=True)
cross_attention = nn.MultiheadAttention(embed_dim=32, num_heads=4, batch_first=True)
tokens = torch.randn(2, 6, 32)
context = torch.randn(2, 10, 32)

self_out, self_weights = self_attention(tokens, tokens, tokens, need_weights=True)
cross_out, cross_weights = cross_attention(tokens, context, context, need_weights=True)
print("Self-attention:", self_out.shape, "weights:", self_weights.shape)
print("Cross-attention:", cross_out.shape, "weights:", cross_weights.shape)

## Comparison: when to reach for each family

| Algorithm | Typical problem | Key idea | Main operational consideration |
|---|---|---|---|
| PPO | Sequential decisions | Clipped policy-gradient update | On-policy data collection can be expensive |
| A* | Route/path planning | Cost-so-far + heuristic | Heuristic quality and state-space size |
| MCTS | Large decision/game trees | Repeated simulation and UCB selection | Compute budget and rollout quality |
| LSTM | Ordered sequences | Gated recurrent state | Sequential computation and long-range memory |
| DNC | Algorithmic reasoning / structured memory | Differentiable external memory | More complex training and memory control |
| Alpha-beta | Adversarial games | Minimax with branch pruning | Move ordering and evaluation quality |
| YOLO | Object detection | One-stage box/class prediction | Model size, latency, confidence thresholds |
| GAN | Synthetic data generation | Generator vs discriminator | Training instability and evaluation |
| VGG / ResNet | Image recognition | Convolutions; residual skips | Compute, memory, transfer-learning choice |
| Behavior trees | Bounded agent/robot control | Composable condition/action nodes | Explicit failure and timeout handling |
| SAC | Continuous control | Off-policy actor-critic + entropy | Replay-buffer and target-network stability |
| Attention | Language and multimodal data | Query-key-value weighted context | Quadratic attention cost for long sequences |

**Production reminder:** treat these as algorithm demonstrations. Real deployments require held-out evaluation, input validation, monitoring, security review, model/version provenance, and workload-specific latency/cost testing.
